# PropAI — Market Intelligence Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/propai/blob/main/notebooks/02_market_analysis.ipynb)

This notebook demonstrates PropAI's market data layer — pulling and synthesizing data from four free sources:

| Source | Data | API Key |
|---|---|---|
| US Census Bureau ACS | Demographics, income, housing | Free (optional) |
| FRED | Mortgage rates, CPI, unemployment | Free (optional) |
| HUD | Fair Market Rents by bedroom | Free, no key |
| Zillow Research | Home values, rent trends | Free, bulk CSV |

All features gracefully degrade — if a key is missing, the rest still runs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'httpx', 'pandas', 'matplotlib', 'asyncio', '-q'], check=True)

import os
if not os.path.exists('propai'):
    os.system('git clone https://github.com/your-org/propai.git --depth=1 -q')
sys.path.insert(0, os.path.join(os.getcwd(), 'propai', 'backend'))
print('Setup complete')

## Configuration

API keys are optional — HUD and Zillow work without any. Census and FRED are free to register for.

In [ ]:
import os

# Set keys here or via environment variables
# os.environ['CENSUS_API_KEY'] = 'your-key-here'
# os.environ['FRED_API_KEY'] = 'your-key-here'

# Target market
TARGET_METRO = 'Austin'
TARGET_STATE_FIPS = '48'   # Texas
TARGET_COUNTY_FIPS = '453' # Travis County
TARGET_ZIP = '78701'

print(f'Target: {TARGET_METRO}, TX  (FIPS {TARGET_STATE_FIPS}-{TARGET_COUNTY_FIPS})')

## 1. FRED — Macroeconomic Snapshot

In [ ]:
import asyncio
from data.fred import FREDClient

async def get_macro():
    async with FREDClient() as fred:
        return await fred.get_macro_snapshot()

macro = asyncio.run(get_macro())

if macro:
    print('FRED Macroeconomic Snapshot')
    print('=' * 45)
    print(f'  30yr Mortgage Rate:   {macro.mortgage_rate_30yr:.2%}')
    print(f'  Fed Funds Rate:       {macro.fed_funds_rate:.2%}')
    print(f'  CPI (YoY):            {macro.cpi_yoy:.1%}')
    print(f'  Unemployment:         {macro.unemployment_rate:.1%}')
    print(f'  10yr Treasury:        {macro.treasury_10yr:.2%}')
    print(f'  GDP Growth:           {macro.gdp_growth:.1%}')
    print(f'  Housing Starts:       {macro.housing_starts:,.0f}')
    print(f'  Rate Environment:     {macro.rate_environment}')
    print(f'  Cap Rate Pressure:    {macro.cap_rate_pressure}')
else:
    print('FRED unavailable (set FRED_API_KEY for live data)')

## 2. HUD — Fair Market Rents

In [ ]:
from data.hud import HUDClient

async def get_fmr():
    async with HUDClient() as hud:
        fips = f'{TARGET_STATE_FIPS}{TARGET_COUNTY_FIPS}'
        return await hud.get_fair_market_rents(fips)

fmr = asyncio.run(get_fmr())

if fmr:
    print(f'HUD Fair Market Rents — {TARGET_METRO}, TX')
    print('=' * 40)
    for br, rent in fmr.by_bedroom.items():
        print(f'  {br:>8s}: ${rent:>7,.0f}/mo')
    print(f'\n  {fmr.rent_growth_commentary()}')
else:
    print('HUD data unavailable for this FIPS code')

## 3. Zillow Research — Home Value & Rent Trends

In [ ]:
from data.zillow import ZillowClient

async def get_zillow():
    async with ZillowClient() as zl:
        return await zl.get_metro_metrics(TARGET_METRO, TARGET_STATE_FIPS)

zillow = asyncio.run(get_zillow())

if zillow:
    print(f'Zillow Research — {TARGET_METRO}')
    print('=' * 45)
    if zillow.median_home_value:
        print(f'  Median Home Value:    ${zillow.median_home_value:>10,.0f}')
        print(f'  ZHVI YoY:             {zillow.zhvi_yoy_pct:>10.1%}')
        print(f'  ZHVI 3yr CAGR:        {zillow.zhvi_3yr_cagr:>10.1%}')
    if zillow.median_asking_rent:
        print(f'  Median Asking Rent:   ${zillow.median_asking_rent:>10,.0f}/mo')
        print(f'  ZORI YoY:             {zillow.zori_yoy_pct:>10.1%}')
    if zillow.price_to_rent_ratio:
        print(f'  Price-to-Rent Ratio:  {zillow.price_to_rent_ratio:>10.1f}x')
    print(f'  Rent Growth Trend:    {zillow.rent_growth_trend or "N/A"}')
else:
    print('Zillow data unavailable for this metro')

## 4. Census Bureau — Demographics

In [ ]:
from data.census import CensusClient

async def get_demographics():
    async with CensusClient() as census:
        return await census.get_county_profile(TARGET_STATE_FIPS, TARGET_COUNTY_FIPS)

demo = asyncio.run(get_demographics())

if demo:
    print(f'Census Demographics — Travis County, TX')
    print('=' * 45)
    print(f'  Total Population:     {demo.total_population:>12,.0f}')
    print(f'  Median HH Income:     ${demo.median_household_income:>11,.0f}')
    print(f'  Median Gross Rent:    ${demo.median_gross_rent:>11,.0f}/mo')
    print(f'  Vacancy Rate:         {demo.vacancy_rate:>12.1%}')
    print(f'  Homeownership Rate:   {demo.homeownership_rate:>12.1%}')
    print(f'  Pct Renter-Occupied:  {demo.pct_renter_occupied:>12.1%}')
    print(f'  Rent-to-Income Ratio: {demo.rent_to_income_ratio:>12.1%}')
    if demo.income_signal:
        print(f'  Income Signal:        {demo.income_signal}')
else:
    print('Census data unavailable (set CENSUS_API_KEY for live data)')

## 5. Full Market Report (All Sources Combined)

In [ ]:
from data.market_service import MarketService

async def get_full_report():
    async with MarketService() as svc:
        return await svc.get_market_report(
            metro=TARGET_METRO,
            state_fips=TARGET_STATE_FIPS,
            county_fips=TARGET_COUNTY_FIPS,
        )

report = asyncio.run(get_full_report())

print(f'\nMarket Report — {TARGET_METRO}')
print('=' * 55)
print(f'  Market Score:          {report.market_score}/100 ({report.market_grade})')
print(f'  Suggested Rent Growth: {report.suggested_rent_growth:.1%}/yr')
print(f'  Exit Cap Range:        {report.suggested_exit_cap_range[0]:.2%} – {report.suggested_exit_cap_range[1]:.2%}')
print()
print('  Tailwinds:')
for t in report.tailwinds:
    print(f'    + {t}')
print('  Headwinds:')
for h in report.headwinds:
    print(f'    – {h}')
print()
print('  Market Thesis:')
print(f'  {report.market_thesis}')

## 6. Market Score Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

score = report.market_score
grade = report.market_grade

fig, ax = plt.subplots(figsize=(8, 3))
fig.patch.set_facecolor('#102a43')
ax.set_facecolor('#102a43')

# Background bar
ax.barh(0, 100, height=0.6, color='#243b53', left=0)

# Color gradient segments
segments = [(25, '#7f1d1d'), (50, '#78350f'), (75, '#065f46'), (100, '#064e3b')]
left = 0
for end, color in segments:
    ax.barh(0, min(score, end) - left, height=0.6, color=color, left=left, alpha=0.9)
    left = end
    if score <= end:
        break

# Score marker
ax.axvline(score, color='#e6b800', linewidth=2.5, ymin=0.1, ymax=0.9)
ax.text(score, 0.5, f' {score}', color='#e6b800', fontweight='bold', fontsize=14, va='center')

ax.set_xlim(0, 100)
ax.set_ylim(-0.5, 1)
ax.set_xlabel('Market Score (0–100)', color='#829ab1')
ax.set_title(f'{TARGET_METRO} Market Grade: {grade}', color='#d9e2ec', fontsize=14, pad=10)
ax.tick_params(colors='#829ab1')
ax.spines[:].set_visible(False)
ax.set_yticks([])

for x, label in [(12, 'D'), (37, 'C'), (62, 'B'), (87, 'A')]:
    ax.text(x, -0.45, label, color='#627d98', fontsize=9, ha='center')

plt.tight_layout()
plt.show()

---
## Next Steps

- **[Notebook 1](./01_underwriting_demo.ipynb)** — Full deal underwriting with IRR, NPV, sensitivity tables
- **[Notebook 3](./03_ai_memo_generation.ipynb)** — Generate an institutional memo with Claude
- **[GitHub](https://github.com/your-org/propai)** — Run the full stack with `docker-compose up`